# 06 · Morning action plan per store, and Unity AI Gateway logs (executed)

`src/jobs/store_daily_briefs.py`: SQL computes every fact (low stock, overdue POs, today's forecast, waste);
`kaapi_llm` through **Unity AI Gateway** ranks and words up to 5 actions. A deterministic check then confirms
every number in each brief appears in its facts; a failed check triggers one rewrite. Each brief is an MLflow trace.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from proof_helpers import *
print('catalog.schema:', FQ)

catalog.schema: fevm_cme_conde_catalog.kaapi_bricks


In [2]:
b = df(f"""SELECT brief_date, store_id, store_name, numbers_checked, numbers_grounded, headline
           FROM {FQ}.store_daily_briefs WHERE brief_date = (SELECT MAX(brief_date) FROM {FQ}.store_daily_briefs)
           ORDER BY store_id""")
b[["numbers_checked", "numbers_grounded"]] = b[["numbers_checked", "numbers_grounded"]].astype(int)
print(f"briefs: {len(b)} stores | numbers verified against facts: {b.numbers_grounded.sum()}/{b.numbers_checked.sum()}")
print(b[["store_id", "store_name", "numbers_grounded", "numbers_checked", "headline"]].head(12).to_string(index=False))

briefs: 37 stores | numbers verified against facts: 882/884
store_id                    store_name  numbers_grounded  numbers_checked                                                                                    headline
 STR-001      Kaapi Bricks Koramangala                23               23            Friday push: chase 8 overdue POs worth ₹31065 and prep for above-average demand.
 STR-002      Kaapi Bricks Indiranagar                26               26                Friday at Indiranagar: clear 7 overdue POs worth ₹49,893 and cut bean waste.
 STR-003        Kaapi Bricks Jayanagar                22               22       Friday: clear overdue POs totalling ₹38,540 and prep for above-average coffee demand.
 STR-004       Kaapi Bricks Whitefield                20               20             Friday at Whitefield: reorder Vanilla Extract, chase overdue POs worth ₹12,865.
 STR-005          Kaapi Bricks MG Road                23               23   Friday at MG Road: chase 6 overdue

In [3]:
one = sql(f"""SELECT store_name, headline, actions_json, facts_json, trace_id FROM {FQ}.store_daily_briefs
             WHERE store_id = 'STR-001' ORDER BY brief_date DESC LIMIT 1""")[0]
print("Koramangala —", one["headline"], "| MLflow trace", one["trace_id"])
for a in json.loads(one["actions_json"]):
    print(f"  {a.get('priority')}. {a['action']}\n     why: {a['why']}")
print("\nFacts the model was given (computed in SQL):")
print(json.dumps(json.loads(one["facts_json"]), indent=1)[:1800])

Koramangala — Friday push: chase 8 overdue POs worth ₹31065 and prep for above-average demand. | MLflow trace tr-937be1218fc3a920259daf418e69542d
  1. Escalate PO-00816 with Mysore Sweet Works; payment of ₹6017 is 182 days overdue.
     why: PO-00816 to Mysore Sweet Works is the oldest overdue at 182 days, risking supplier relationship.
  2. Follow up PO-00951 with Mysore Sweet Works for ₹4967 overdue 136 days.
     why: Second overdue PO to same supplier; 8 total overdue POs worth ₹31065 need urgent resolution.
  3. Chase PO-01314 with Coorg Coffee Estates for ₹6341 overdue 130 days.
     why: PO-01314 is 130 days overdue and remains unresolved.
  4. Prep extra Classic Filter Coffee and Masala Chai batches for today's higher-than-usual demand.
     why: Classic Filter forecast 14.1 vs usual 12.3; Masala Chai forecast 9.8 vs usual 7.3 on Friday.
  5. Review Chikmagalur Robusta Beans prep quantities to cut ₹4544 waste this week.
     why: Chikmagalur Robusta Beans is the top waste item 

## Every model call is in the gateway payload log

In [4]:
# requester = the service principal that called the gateway; resolve it to a readable name.
names = {hq.apps.get(a).service_principal_client_id: f"app {a}" for a in ("kaapi-bricks-finale", "kaapi-ops-mcp")}
names[hq.current_user.me().user_name] = "ops job / notebook (HQ user)"
for t in ("kaapi_llm_payload", "kaapi_embed_payload"):
    x = df(f"""SELECT requester, destination_model, COUNT(*) AS requests,
               SUM(CASE WHEN status_code = 200 THEN 1 ELSE 0 END) AS ok,
               CAST(percentile(latency_ms, 0.5) AS INT) AS median_latency_ms, MAX(event_time) AS last_call
               FROM {FQ}.{t} GROUP BY 1, 2 ORDER BY requests DESC""")
    x["requester"] = x.requester.map(lambda r: names.get(r, "other principal"))
    print(t); print(x.to_string(index=False)); print()

kaapi_llm_payload
                   requester destination_model requests  ok median_latency_ms                last_call
ops job / notebook (HQ user)           primary      104 104              3956 2026-10-23T18:08:17.472Z
     app kaapi-bricks-finale           primary        3   3              3828 2026-10-23T18:14:38.635Z
           app kaapi-ops-mcp           primary        2   2             21876 2026-10-23T18:14:55.838Z
ops job / notebook (HQ user)               NaN        2   0                 0 2026-09-29T12:24:17.308Z



kaapi_embed_payload
              requester destination_model requests ok median_latency_ms                last_call
app kaapi-bricks-finale           primary        4  4               666 2026-10-06T06:58:09.833Z

